# Model Registry & Versioning

Managing models through their lifecycle: development -> staging -> production.

1. **MLflow Model Registry** - Register, version, and stage models
2. **Model Versioning** - Track which model version is in production
3. **Model Signatures** - Define input/output schemas
4. **Reproducibility** - Pinning dependencies and environment

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mlflow.set_experiment("model_registry_demo")

## 1. Register a Model with Signature

In [ ]:
with mlflow.start_run(run_name="registry_demo_v1"):
    # Train
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    
    # Predict for signature
    y_pred = pipe.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Infer model signature (input/output schema)
    signature = infer_signature(X_test, y_pred)
    print(f"Model Signature:\n{signature}")
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_param("n_estimators", 100)
    
    # Log and register model in one step
    mlflow.sklearn.log_model(
        pipe,
        artifact_path="model",
        signature=signature,
        registered_model_name="breast_cancer_classifier",  # This registers the model
    )
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print("Model registered as 'breast_cancer_classifier' v1")

## 2. Model Lifecycle Stages

```
Development -> Staging -> Production -> Archived
```

```python
from mlflow import MlflowClient

client = MlflowClient()

# Transition model version to staging
client.transition_model_version_stage(
    name="breast_cancer_classifier",
    version=1,
    stage="Staging"
)

# After validation, promote to production
client.transition_model_version_stage(
    name="breast_cancer_classifier",
    version=1,
    stage="Production"
)

# Load production model
model = mlflow.sklearn.load_model("models:/breast_cancer_classifier/Production")
```

## 3. Model Versioning Workflow

```python
# Train a new version
with mlflow.start_run():
    # ... train improved model ...
    mlflow.sklearn.log_model(
        improved_pipe,
        "model",
        registered_model_name="breast_cancer_classifier",  # Same name = new version
    )
    # This creates version 2 automatically

# Compare versions
v1 = mlflow.sklearn.load_model("models:/breast_cancer_classifier/1")
v2 = mlflow.sklearn.load_model("models:/breast_cancer_classifier/2")

# A/B test or shadow test before promoting
```

## Best Practices for Model Versioning

| Practice | Why |
|----------|-----|
| Always log model signatures | Catch input schema mismatches before production |
| Use staging before production | Validate on shadow traffic or hold-out data |
| Tag models with metadata | Training dataset version, git commit hash |
| Never delete production models | Archive them instead for rollback capability |
| Pin dependency versions | `conda.yaml` or `requirements.txt` in model artifact |

## Key Takeaways

1. **Model Registry is your single source of truth** for deployed models
2. **Signatures validate inputs** - prevent serving errors from schema mismatches
3. **Version everything** - model code, training data, hyperparameters
4. **Staging -> Production** workflow prevents deploying untested models
5. **MLflow stores environment info** (conda.yaml) for reproducible deployment